# Student Performance Prediction System

## Model Training & Evaluation

### CodeVedX AI/ML Internship - Project 2

---

**Objective:**
This notebook performs the complete Machine Learning workflow:
load processed data, encode categorical features, split data,
train multiple regression models, evaluate them, select the best
model, and save it along with evaluation artefacts.

**Models trained:**
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

**Evaluation metrics:** MAE, MSE, RMSE, R-squared.

**Outputs:**
- Trained model: models/student_performance_model.pkl
- Evaluation charts: outputs/charts/
- Evaluation report: outputs/reports/


In [ ]:
# ========================================
# STEP 1: Import Libraries
# ========================================

import warnings
warnings.filterwarnings('ignore')

import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Regressors
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Model persistence
import joblib

print(f'Python    : {sys.version.split()[0]}')
print(f'Pandas    : {pd.__version__}')
print(f'NumPy     : {np.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')
print(f'Scikit-learn: imported')
print(f'Joblib    : imported')
print('\nAll libraries imported successfully.')


In [ ]:
# ========================================
# STEP 2: Configure Display & Style
# ========================================

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 150,
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

PALETTE = sns.color_palette('viridis', 10)

print('Display and style configured.')


In [ ]:
# ========================================
# STEP 3: Setup Project Paths
# ========================================

PROJECT_ROOT = Path('..')
PROCESSED_DATA = PROJECT_ROOT / 'data' / 'processed' / 'student_performance.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
CHARTS_DIR = PROJECT_ROOT / 'outputs' / 'charts'
REPORTS_DIR = PROJECT_ROOT / 'outputs' / 'reports'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

MODEL_PATH = MODELS_DIR / 'student_performance_model.pkl'
REPORT_PATH = REPORTS_DIR / 'model_evaluation_report.json'

print(f'  Processed data : {PROCESSED_DATA}')
print(f'  Model output   : {MODEL_PATH}')
print(f'  Charts         : {CHARTS_DIR}')
print(f'  Report         : {REPORT_PATH}')


In [ ]:
# ========================================
# STEP 4: Load Processed Dataset
# ========================================

print('=' * 60)
print('LOAD PROCESSED DATASET')
print('=' * 60)

df = pd.read_csv(PROCESSED_DATA)
print(f'  Rows   : {df.shape[0]:,}')
print(f'  Columns: {df.shape[1]}')
print(f'  Missing: {df.isnull().sum().sum()}')
print(f'  Dup.   : {df.duplicated().sum()}')

df.head(3)


In [ ]:
# ========================================
# STEP 5: Separate Features & Target
# ========================================

print('=' * 60)
print('FEATURES & TARGET')
print('=' * 60)

target = 'Exam_Score'
X = df.drop(columns=[target])
y = df[target]

print(f'  Features (X) : {X.shape[1]} columns, {X.shape[0]:,} rows')
print(f'  Target  (y) : {y.name}, {len(y):,} values')
print(f'  Target range: {y.min():.0f} - {y.max():.0f}')

# Identify column types
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f'\n  Numerical features : {len(num_features)}')
print(f'  Categorical features: {len(cat_features)}')
print(f'\n  Numerical: {num_features}')
print(f'  Categorical: {cat_features}')


In [ ]:
# ========================================
# STEP 6: Encode Categorical Variables
# ========================================

print('=' * 60)
print('ENCODE CATEGORICAL VARIABLES')
print('=' * 60)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ])

print(f'  Numerical transformer: StandardScaler ({len(num_features)} cols)')
print(f'  Categorical transformer: OneHotEncoder (drop=first) ({len(cat_features)} cols)')

# Preview transformed shape
X_encoded = preprocessor.fit_transform(X)
feature_names = (num_features + 
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features)))
print(f'\n  After encoding: {X_encoded.shape[1]} features')


In [ ]:
# ========================================
# STEP 7: Split Dataset (80/20)
# ========================================

print('=' * 60)
print('TRAIN-TEST SPLIT')
print('=' * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42)

print(f'  Training set   : {X_train.shape[0]:,} samples ({100*0.8:.0f}%)')
print(f'  Testing set    : {X_test.shape[0]:,} samples (20%)')
print(f'  Random state   : 42')


In [ ]:
# ========================================
# STEP 8: Define Training & Evaluation Functions
# ========================================

def train_and_evaluate(model, model_name, X_train, X_test, y_train, y_test):
    """Train a regression model and return metrics + predictions."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    return {
        'Model': model_name,
        'MAE': round(mae, 4),
        'MSE': round(mse, 4),
        'RMSE': round(rmse, 4),
        'R2': round(r2, 4)
    }, y_pred, model

print('Training and evaluation function defined.')


In [ ]:
# ========================================
# STEP 9: Train Models
# ========================================

print('=' * 60)
print('TRAINING REGRESSION MODELS')
print('=' * 60)

models = [
    ('Linear Regression', LinearRegression()),
    ('Decision Tree', DecisionTreeRegressor(random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42, n_estimators=100))
]

results = []
predictions = {}
trained_models = {}

for name, model_obj in models:
    print(f'\nTraining {name}...')
    metrics, y_pred, trained = train_and_evaluate(
        model_obj, name, X_train, X_test, y_train, y_test)
    results.append(metrics)
    predictions[name] = y_pred
    trained_models[name] = trained
    print(f'  R-squared: {metrics["R2"]:.4f}')
    print(f'  RMSE     : {metrics["RMSE"]:.4f}')

print('\nAll models trained successfully.')


In [ ]:
# ========================================
# STEP 10: Model Comparison Table
# ========================================

print('=' * 60)
print('MODEL COMPARISON')
print('=' * 60)

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
display(results_df)

# Save to CSV for reference
results_df.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)
print('Comparison table saved: outputs/reports/model_comparison.csv')


In [ ]:
# ========================================
# STEP 11: Select Best Model
# ========================================

print('=' * 60)
print('BEST MODEL SELECTION')
print('=' * 60)

best_row = results_df.iloc[0]
best_model_name = best_row['Model']
best_model = trained_models[best_model_name]

print(f'\n  Best model: {best_model_name}')
print(f'  R-squared : {best_row["R2"]:.4f}')
print(f'  RMSE      : {best_row["RMSE"]:.4f}')
print(f'  MAE       : {best_row["MAE"]:.4f}')

print(f'\n  Reason for selection:')
print(f'  The {best_model_name} achieves the highest R-squared value')
print(f'  and the lowest error metrics among all trained models,')
print(f'  indicating it explains the most variance in Exam_Score')
print(f'  and makes the most accurate predictions.')


In [ ]:
# ========================================
# STEP 12: Save Best Model
# ========================================

print('=' * 60)
print('SAVE BEST MODEL')
print('=' * 60)

# Save the full pipeline (preprocessor + model)
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', best_model)
])

# Re-train pipeline on full data
final_pipeline.fit(X, y)

# Save with joblib
joblib.dump(final_pipeline, MODEL_PATH)

print(f'  Model saved: {MODEL_PATH}')
print(f'  File size  : {os.path.getsize(MODEL_PATH) / 1024**2:.2f} MB')
print(f'  Pipeline   : StandardScaler + OneHotEncoder + {best_model_name}')


In [ ]:
# ========================================
# STEP 13: Generate Predictions (Test Set)
# ========================================

print('=' * 60)
print('GENERATE PREDICTIONS')
print('=' * 60)

y_pred_best = predictions[best_model_name]

# Create a comparison DataFrame
comparison_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred_best,
    'Residual': y_test.values - y_pred_best
})
comparison_df['Abs_Error'] = np.abs(comparison_df['Residual'])

print(comparison_df.head(10))
print(f'\n  Mean Absolute Error: {comparison_df["Abs_Error"].mean():.4f}')
print(f'  Max Residual       : {comparison_df["Residual"].max():.4f}')
print(f'  Min Residual       : {comparison_df["Residual"].min():.4f}')


In [ ]:
# ========================================
# STEP 14: Actual vs Predicted Scatter
# ========================================

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.4, color=PALETTE[0])

# Identity line
min_val = min(y_test.min(), y_pred_best.min())
max_val = max(y_test.max(), y_pred_best.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.title(f'Actual vs Predicted Exam Score ({best_model_name})', fontsize=14, fontweight='bold')
plt.xlabel('Actual Exam Score')
plt.ylabel('Predicted Exam Score')
plt.legend()
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'actual_vs_predicted.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: actual_vs_predicted.png')


In [ ]:
# ========================================
# STEP 15: Residual Plot
# ========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual vs Predicted
axes[0].scatter(y_pred_best, comparison_df['Residual'], alpha=0.4, color=PALETTE[1])
axes[0].axhline(y=0, color='red', ls='--', lw=2)
axes[0].set_title('Residuals vs Predicted Values', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Exam Score')
axes[0].set_ylabel('Residual (Actual - Predicted)')

# Residual distribution
sns.histplot(comparison_df['Residual'], kde=True, bins=30, color=PALETTE[2], ax=axes[1])
axes[1].axvline(x=0, color='red', ls='--', lw=2)
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'residual_plot.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: residual_plot.png')


In [ ]:
# ========================================
# STEP 16: Prediction Error Distribution
# ========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute error distribution
sns.histplot(comparison_df['Abs_Error'], kde=True, bins=30, color=PALETTE[3], ax=axes[0])
axes[0].set_title('Absolute Prediction Error Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Absolute Error')
axes[0].set_ylabel('Frequency')

# Error boxplot
sns.boxplot(y=comparison_df['Abs_Error'], color=PALETTE[4], ax=axes[1])
axes[1].set_title('Absolute Error Boxplot', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Absolute Error')

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'error_distribution.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: error_distribution.png')


In [ ]:
# ========================================
# STEP 17: Feature Importance (if supported)
# ========================================

print('=' * 60)
print('FEATURE IMPORTANCE ANALYSIS')
print('=' * 60)

# Check if the best model has feature_importances_
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    f_names = (num_features + 
        list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features)))

    # Create DataFrame and sort
    fi_df = pd.DataFrame({'Feature': f_names, 'Importance': importances})
    fi_df = fi_df.sort_values('Importance', ascending=True)

    # Plot top 15
    top_n = min(15, len(fi_df))
    plt.figure(figsize=(10, 8))
    plt.barh(fi_df['Feature'].tail(top_n), fi_df['Importance'].tail(top_n), color=PALETTE[5])
    plt.title(f'Top {top_n} Feature Importances ({best_model_name})', fontsize=14, fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig(str(CHARTS_DIR / 'feature_importance.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print('Chart saved: feature_importance.png')
    print('\nTop 10 features:')
    print(fi_df.tail(10).to_string(index=False))
else:
    print(f'{best_model_name} does not support native feature importance.')
    print('Skipping feature importance plot.')


In [ ]:
# ========================================
# STEP 18: Model Comparison Bar Chart
# ========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R-squared comparison
colors = ['#2ecc71' if m == best_model_name else '#3498db' for m in results_df['Model']]
axes[0].bar(results_df['Model'], results_df['R2'], color=colors)
axes[0].set_title('R-squared Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylabel('R-squared')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)

# RMSE comparison
axes[1].bar(results_df['Model'], results_df['RMSE'], color=colors)
axes[1].set_title('RMSE Comparison', fontsize=13, fontweight='bold')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'model_comparison.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: model_comparison.png')


In [ ]:
# ========================================
# STEP 19: Generate Evaluation Report
# ========================================

print('=' * 60)
print('GENERATE EVALUATION REPORT')
print('=' * 60)

report = {
    'project': 'Student Performance Prediction System',
    'best_model': best_model_name,
    'metrics': {
        'R2': best_row['R2'],
        'MAE': best_row['MAE'],
        'MSE': best_row['MSE'],
        'RMSE': best_row['RMSE']
    },
    'all_models': results,
    'dataset': {
        'rows': len(df),
        'features': X.shape[1],
        'numerical': len(num_features),
        'categorical': len(cat_features)
    },
    'train_test_split': {
        'train_size': int(X_train.shape[0]),
        'test_size': int(X_test.shape[0]),
        'random_state': 42
    },
    'mean_absolute_error': float(comparison_df['Abs_Error'].mean())
}

with open(REPORT_PATH, 'w') as f:
    json.dump(report, f, indent=2)

print(f'  Report saved: {REPORT_PATH}')
print('\nReport contents:')
print(json.dumps(report, indent=2))


In [ ]:
# ========================================
# STEP 20: Summary
# ========================================

print('=' * 60)
print('MODEL TRAINING COMPLETE')
print('=' * 60)

print()
print('    Clean processed dataset')
print('    Categorical features encoded')
print('    Dataset split (80/20)')
print(f'    Models trained: {len(models)}')
for r in results:
    print(f'      - {r["Model"]:20s}: R2={r["R2"]:.4f}, RMSE={r["RMSE"]:.4f}')
print(f'    Best model: {best_model_name}')
print(f'    Model saved: {MODEL_PATH.name}')
print('    Charts saved to outputs/charts/')
print('    Evaluation report saved to outputs/reports/')

print()
print('=' * 60)
print('  MACHINE LEARNING WORKFLOW COMPLETED SUCCESSFULLY')
print('=' * 60)
